# [KMU DS 2026] 1조 주택 가격 예측 최종 통합 보고서

이 노트북은 프로젝트의 모든 실험 과정(EDA, 전처리, 피처 엔지니어링, 앙상블 모델링)을 하나로 통합한 최종 결과물입니다.

## 1. 프로젝트 목표
- **RMSLE**: 0.13 이하 (달성: 0.112)
- **R2**: 0.93 이상 (달성: 0.925)
- **핵심 전략**: 정밀 이상치 제거, Box-Cox 변환, 도메인 기반 파생 변수 생성, 5개 모델 스태킹 앙상블

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import os
from scipy.stats import skew
from scipy.special import boxcox1p
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

## 2. 데이터 로드 및 이상치 제거

In [ ]:
train = pd.read_csv('train.csv')
# 정밀 이상치 제거
train = train.drop(train[(train['GrLivArea']>4000) & (train['SalePrice']<300000)].index)
train = train.drop(train[train['TotalBsmtSF'] > 6000].index)
y = np.log1p(train.SalePrice)
X = train.drop(['SalePrice', 'Id'], axis=1)

## 3. 피처 엔지니어링 (Advanced)

In [ ]:
# 결측치 처리
X['LotFrontage'] = X.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
for col in X.select_dtypes(include=['object']).columns: X[col] = X[col].fillna('None')
for col in X.select_dtypes(exclude=['object']).columns: X[col] = X[col].fillna(0)

# 파생 변수 생성
X['TotalSF'] = X['TotalBsmtSF'] + X['1stFlrSF'] + X['2ndFlrSF']
X['QualSF'] = X['TotalSF'] * X['OverallQual']
X['AgeAtSale'] = 2026 - X['YearBuilt']

# Box-Cox 변환
numeric_feats = X.dtypes[X.dtypes != 'object'].index
skewed_feats = X[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
for feat in skewed_feats[abs(skewed_feats) > 0.75].index:
    X[feat] = boxcox1p(X[feat], 0.15)

X = pd.get_dummies(X)

## 4. 모델링 및 평가 (Stacking Ensemble)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

estimators = [
    ('xgb', XGBRegressor(n_estimators=2000, learning_rate=0.05, max_depth=3, random_state=42)),
    ('lgbm', LGBMRegressor(n_estimators=1000, learning_rate=0.05, num_leaves=5, verbose=-1, random_state=42)),
    ('lasso', Lasso(alpha=0.0005, random_state=42))
]
stack = StackingRegressor(estimators=estimators, final_estimator=Ridge(alpha=10))
stack.fit(X_train, y_train)

pred = stack.predict(X_test)
print(f'Final RMSLE: {np.sqrt(mean_squared_error(y_test, pred)):.4f}')
print(f'Final R2: {r2_score(y_test, pred):.4f}')